# i-PROGNOSIS typing experiment

Run cells from top to bottom. The notebook exposes dataset/audit inspection and output paths; reusable parsing, feature extraction, and training logic remain in `src/iprognosis_data.py` and `src/train_typing_models.py`.

In [1]:
from pathlib import Path
import sys

# Find the repository root so package imports and relative artifact paths work from either launch location.
PROJECT_ROOT = Path.cwd()
# If Jupyter started inside notebooks/, move up one folder to the repository root.
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'src').is_dir(), 'Open this notebook from the repository or notebooks directory'
# Put the repository root on Python's import path for the src package.
sys.path.insert(0, str(PROJECT_ROOT))

from src.iprognosis_data import DEFAULT_IPROGNOSIS_ROOT, build_participant_dataset
from src.train_typing_models import run_experiment

# Point to the local event files and clinical label CSV.
DATA_ROOT = DEFAULT_IPROGNOSIS_ROOT
MODEL_PATH = PROJECT_ROOT / 'models' / 'finalization_baseline' / 'typing_iprognosis_model.joblib'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'finalization_baseline' / 'typing'
# Use the documented fixed seed for the participant split and model searches.
SEED = 42
print('Typing data root:', DATA_ROOT)
print('Data directory exists:', (DATA_ROOT / 'Data').is_dir())
print('Clinical CSV exists:', (DATA_ROOT / 'Demographics_Clinical_Characteristics.csv').is_file())

Typing data root: C:\Users\daisy\Projects\AI\pd-ml-lab\data\virtual keyboard data\iprognosis_typing_dev
Data directory exists: True
Clinical CSV exists: True


In [2]:
# Parse/link the local cohort and inspect participant labels and data-quality audit before training.
participants, excerpt_features, data_audit = build_participant_dataset(DATA_ROOT)
display(participants.groupby('target').size().rename('participants').to_frame())
display(participants.head())
display(excerpt_features.head())
data_audit

,participants
target,
0,15
1,18


,participant_id,target,hold_ms_mean,hold_ms_std,hold_ms_median,hold_ms_q90,hold_ms_iqr,hold_ms_cv,hold_ms_q10,hold_ms_skew,...,pressure_normalized_skew,pressure_normalized_kurtosis,pause_fraction,press_interval_ms_acf1,hold_ms_acf1,hold_ms_median_trend,press_interval_ms_median_trend,pause_fraction_trend,excerpt_count,total_valid_events
0,01,1,80.690909,17.538789,80.0,104.8,24.000,0.219043,61.00,0.218078,...,-0.657715,-0.298183,0.335052,-0.005816,0.108567,-0.381818,-4.481818,-0.001267,11,782
1,02,1,117.818182,23.108837,116.0,145.6,28.500,0.203190,89.80,0.026398,...,-1.031615,0.996358,0.539216,-0.081524,0.111909,-1.709091,-24.577273,-0.008627,11,1085
2,03,1,101.079208,31.278153,99.0,129.0,29.500,0.302652,68.70,1.555260,...,-0.615772,0.074208,0.580645,-0.077964,0.016513,-0.172727,-71.177273,-0.013918,11,963
3,04,1,161.150685,25.780965,160.0,188.8,31.000,0.172436,131.80,0.390742,...,-0.504482,1.134332,0.119048,0.043885,0.094615,-2.990909,2.709091,0.000992,11,993
4,05,1,78.288462,22.070318,76.0,103.7,22.875,0.274266,53.65,0.262894,...,-0.303177,0.388346,0.188948,0.056801,0.077347,-1.245455,-5.203030,-0.003337,10,756


,participant_id,excerpt_id,target,event_count_audit_only,hold_ms_mean,hold_ms_std,hold_ms_median,hold_ms_q90,hold_ms_iqr,hold_ms_cv,...,pressure_normalized_median,pressure_normalized_q90,pressure_normalized_iqr,pressure_normalized_cv,pressure_normalized_q10,pressure_normalized_skew,pressure_normalized_kurtosis,pause_fraction,press_interval_ms_acf1,hold_ms_acf1
0,01,1,1,195,86.235897,20.543780,86.0,113.0,27.0,0.238228,...,0.5625,0.6250,0.1750,0.201561,0.3750,-0.657715,0.109501,0.335052,-0.080086,0.237495
1,01,2,1,49,85.224490,17.182413,83.0,107.2,24.0,0.201614,...,0.5000,0.6250,0.1750,0.197954,0.3600,-0.059440,-1.131721,0.375000,-0.103014,-0.078096
2,01,3,1,57,75.614035,16.831210,74.0,93.6,18.0,0.222594,...,0.5750,0.6250,0.1375,0.187641,0.3825,-0.820543,-0.357545,0.339286,-0.005816,0.116336
3,01,4,1,47,90.510638,16.439279,90.0,109.8,19.0,0.181628,...,0.5750,0.6425,0.1750,0.166541,0.4000,-0.416698,-1.225488,0.326087,-0.147354,0.074057
4,01,5,1,57,80.070175,17.538789,81.0,98.4,19.0,0.219043,...,0.5750,0.6500,0.1750,0.219700,0.3875,-1.210141,1.287818,0.357143,0.145234,0.108567


{'dataset': 'i-PROGNOSIS touchscreen typing development dataset',
 'label_source': 'Group column in Demographics_Clinical_Characteristics.csv; Control=0, PD=1',
 'participant_id_source': 'S## folder and S##_TEX## filename linked to Subject ID column',
 'event_schema': ['Press',
  'press timestamp milliseconds',
  'Release timestamp milliseconds',
  'normalized pressure 0-1'],
 'timestamp_unit': 'milliseconds, as documented in local ReadMe.txt',
 'pressure_summary_policy': 'Only observed values in documented [0,1] range enter pressure features; all values are retained in audit counts, no clipping',
 'aggregation': 'median of excerpt-level feature summaries per participant, plus per-participant slope of selected excerpt medians across excerpts (*_trend); event and excerpt counts are audit-only',
 'participants': 33,
 'participants_by_label': {'control': 15, 'pd': 18},
 'excerpt_files': 338,
 'excerpt_files_by_participant_min': 5,
 'excerpt_files_by_participant_max': 11,
 'audit_totals': 

In [3]:
# Run participant holdout, training-only grouped CV, selection, test evaluation, and artifact writing.
summary = run_experiment(
    iprognosis_root=DATA_ROOT,
    model_path=MODEL_PATH,
    results_dir=RESULTS_DIR,
    seed=SEED,
)
summary

Nested CV: participant|all|logistic_regression (51 features, 33 rows)
Nested CV: participant|all|logistic_l1 (51 features, 33 rows)
Nested CV: participant|all|random_forest (51 features, 33 rows)
Nested CV: participant|no_pressure|logistic_regression (42 features, 33 rows)
Nested CV: participant|no_pressure|logistic_l1 (42 features, 33 rows)
Nested CV: participant|no_pressure|random_forest (42 features, 33 rows)
Nested CV: participant|core|logistic_regression (8 features, 33 rows)
Nested CV: participant|core|logistic_l1 (8 features, 33 rows)
Nested CV: participant|core|random_forest (8 features, 33 rows)
Nested CV: excerpt|all|logistic_regression (48 features, 338 rows)
Nested CV: excerpt|all|logistic_l1 (48 features, 338 rows)
Nested CV: excerpt|all|random_forest (48 features, 338 rows)
Nested CV: excerpt|no_pressure|logistic_regression (39 features, 338 rows)
Nested CV: excerpt|no_pressure|logistic_l1 (39 features, 338 rows)
Nested CV: excerpt|no_pressure|random_forest (39 features, 

{'study_scope': 'Retrospective classification of known PD versus control status from smartphone touchscreen typing; not prospective risk prediction',
 'dataset_audit': {'dataset': 'i-PROGNOSIS touchscreen typing development dataset',
  'label_source': 'Group column in Demographics_Clinical_Characteristics.csv; Control=0, PD=1',
  'participant_id_source': 'S## folder and S##_TEX## filename linked to Subject ID column',
  'event_schema': ['Press',
   'press timestamp milliseconds',
   'Release timestamp milliseconds',
   'normalized pressure 0-1'],
  'timestamp_unit': 'milliseconds, as documented in local ReadMe.txt',
  'pressure_summary_policy': 'Only observed values in documented [0,1] range enter pressure features; all values are retained in audit counts, no clipping',
  'aggregation': 'median of excerpt-level feature summaries per participant, plus per-participant slope of selected excerpt medians across excerpts (*_trend); event and excerpt counts are audit-only',
  'participants': 

In [4]:
print('Model:', MODEL_PATH, MODEL_PATH.exists())
print('Results:', sorted(p.name for p in RESULTS_DIR.iterdir()))

Model: c:\Users\daisy\Projects\AI\pd-ml-lab\models\finalization_baseline\typing_iprognosis_model.joblib True
Results: ['typing_cv_results.csv', 'typing_excerpt_feature_audit.csv', 'typing_experiment_report.md', 'typing_feature_stability.csv', 'typing_oof_predictions.csv', 'typing_participant_features.csv', 'typing_split_assignments.csv', 'typing_test_predictions.csv', 'typing_training_config.json', 'typing_training_metrics.json']
